In [79]:
import random
import statistics
import sys
import os

In [80]:
import math

class KNNScratch:
  def __init__(self,k=3):
    self.k = k
    self.X_train = []
    self.y_train = []
    
  def fit(self, X, y):
    
    """
    KNN is a 'Lazy Learner'. 
    Fit just means 'memorize the data'.
    """
    
    self.X_train = X
    self.y_train = y
    
  def _euclidean_distance(self, x1, x2):
    distance = 0.0
    for i in range(len(x1)):
      distance += (x1[i] - x2[i]) ** 2
    return math.sqrt(distance)
    
  def predict(self, X):
    predictions = []
    for x_new in X:
      # Step 1: Calculate distances to ALL training points
      distances = []
      for i in range(len(self.X_train)):
        dist = self._euclidean_distance(x_new, self.X_train[i])
        # We store a tuple: (distance, actual_label)
        distances.append((dist, self.y_train[i]))
        
      # Step 2: Sort by distance (first item in tuple)
      distances.sort(key=lambda x: x[0])
      
      # Step 3: Get K nearest neighbors
      k_nearest = distances[:self.k['k']]
      
      # Step 4: Majority Vote
      # Extract just the labels from the k_nearest
      k_labels = [label for _, label in k_nearest]
          
      most_common = max(set(k_labels), key=k_labels.count)
      predictions.append(most_common)
      
    return predictions

In [81]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

def plot_decision_boundary(model, X, y):
    # Create color maps
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
    cmap_bold = ListedColormap(['#FF0000', '#00FF00'])

    # Calculate min/max boundaries for the plot
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    
    # Generate a grid of points (the "mesh")
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
    np.arange(y_min, y_max, 0.1))

    # Predict every point in the grid
    # We use np.c_ to stack them into (x,y) pairs
    Z = np.array(model.predict(np.c_[xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)

    # Plot the result
    plt.figure(figsize=(8, 6))
    plt.pcolormesh(xx, yy, Z, cmap=cmap_light, shading='auto')

    # Plot the original training points
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, edgecolor='k', s=20)
    plt.title(f"KNN Decision Boundary (k={model.k})")
    plt.show()

In [82]:
def accuracy_score(y_true, y_pred):
    correct = 0
    for yt, yp in zip(y_true, y_pred):
        if yt == yp:
            correct += 1
    return correct / len(y_true)

# CHANGED: Renamed 'k' to 'folds' to avoid conflict with KNN's 'k'
def cross_validation(model_class, X, y, folds=5, **model_params):
    """
    model_class: The Class name (e.g., KNNScratch)
    X, y: Lists of data
    folds: Number of cross-validation splits
    model_params: Arguments for the model (e.g., k=3, learning_rate=0.01)
    """
    n_samples = len(X)
    fold_size = n_samples // folds
    
    indices = list(range(n_samples))
    random.shuffle(indices)
    
    scores = []
    print(f"--- Starting {folds}-Fold Cross Validation ---")

    for i in range(folds):
        start = i * fold_size
        end = (i + 1) * fold_size if i != folds - 1 else n_samples
        
        test_idx_set = set(indices[start:end])
        train_idx_set = [idx for idx in indices if idx not in test_idx_set]
        
        X_train_fold = [X[idx] for idx in train_idx_set]
        y_train_fold = [y[idx] for idx in train_idx_set]
        X_test_fold  = [X[idx] for idx in test_idx_set]
        y_test_fold  = [y[idx] for idx in test_idx_set]
        
        # Instantiate model with model_params (e.g., k=3)
        model = model_class(model_params)
        model.fit(X_train_fold, y_train_fold)
        
        predictions = model.predict(X_test_fold)
        
        score = accuracy_score(y_test_fold, predictions)
        scores.append(score)
        
        print(f"Fold {i+1}: Accuracy = {score*100:.1f}%")

    mean_score = statistics.mean(scores)
    std_dev = statistics.stdev(scores)
    
    print(f"--------------------------------")
    print(f"Average Accuracy: {mean_score*100:.2f}% (+/- {std_dev*100:.2f}%)")
    return mean_score

In [87]:
if __name__ == "__main__":
  # Let's recreate the dummy data from KNN module
  # Class 0: Small numbers, Class 1: Large numbers
  from sklearn.datasets import make_moons
  X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

  cross_validation(KNNScratch, X, y, folds=10, k=1)

--- Starting 10-Fold Cross Validation ---
Fold 1: Accuracy = 98.0%
Fold 2: Accuracy = 94.0%
Fold 3: Accuracy = 97.0%
Fold 4: Accuracy = 97.0%
Fold 5: Accuracy = 96.0%
Fold 6: Accuracy = 92.0%
Fold 7: Accuracy = 97.0%
Fold 8: Accuracy = 96.0%
Fold 9: Accuracy = 97.0%
Fold 10: Accuracy = 97.0%
--------------------------------
Average Accuracy: 96.10% (+/- 1.79%)
